# Taller 1 — Procesamiento de datos olímpicos con Apache Spark

**Curso:** Gestión de datos y tecnologías Big Data
**Grupo:** 1
**Integrantes:** Yerko Gallardo, Sebastián, José Ignacio Rodríguez

**Objetivo:** procesar el registro histórico de deportistas y resultados olímpicos con PySpark, utilizando RDDs, DataFrames y Spark SQL.

**Fuente de datos:** archivos entregados en el curso, ubicados en la carpeta `data/` del proyecto (`deportista.csv`, `deportista2.csv`, `resultados.csv`, `evento.csv`, `equipo.csv`, `juegos.json`).

## 1. Configuración del entorno

El proyecto se gestiona con **uv**: la versión de Python y los paquetes quedan fijados en `pyproject.toml`, `uv.lock` y `.python-version`, lo que permite reproducir el mismo entorno en cualquier equipo.
**Java** es una dependencia de ejecución de Spark (el motor corre en la JVM) y se instala fuera de uv.

| Componente | Versión |
|---|---|
| Python | 3.14 |
| PySpark | 4.3.0 |
| Java (JDK) | 21 (Eclipse Temurin) |

**Preparación (macOS):**

```bash
# Java
brew install --cask temurin@21
echo 'export JAVA_HOME=$(/usr/libexec/java_home -v 21)' >> ~/.zshrc
source ~/.zshrc
java -version

# Proyecto
git clone https://github.com/joserodriguezc/gestiondatosytecnologiasbigdata.git
cd gestiondatosytecnologiasbigdata
uv sync --locked

# Editor
code .        # VS Code: seleccionar el kernel .venv (Python 3.14)
# alternativa: uv run jupyter lab
```

La sesión se crea en modo clásico (no Spark Connect), ya que el taller requiere RDDs y `SparkContext`.

### 1.1 Verificación del intérprete, versiones y Java

In [ ]:
import os, sys, platform, subprocess
import pyspark

# Si el editor no heredó JAVA_HOME, se obtiene del sistema (macOS) sin fijar rutas personales
if not os.environ.get("JAVA_HOME") and platform.system() == "Darwin":
    os.environ["JAVA_HOME"] = subprocess.run(
        ["/usr/libexec/java_home", "-v", "21"], capture_output=True, text=True
    ).stdout.strip()

print("Sistema    :", platform.system(), platform.machine())
print("Intérprete :", sys.executable)
print("Python     :", platform.python_version())
print("PySpark    :", pyspark.__version__)
print("JAVA_HOME  :", os.environ.get("JAVA_HOME", "NO DEFINIDO"))

java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java       :")
print(java.stderr.strip())   # java -version escribe en stderr

### 1.2 SparkSession y SparkContext

`PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al intérprete del proyecto para que los procesos Python de Spark usen el mismo entorno.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Taller1_Spark")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

print("Tipo de sesión     :", type(spark).__module__)   # pyspark.sql.session = modo clásico
print("Versión Spark      :", spark.version)
print("Master             :", sc.master)
print("Nombre aplicación  :", sc.appName)
print("Paralelismo default:", sc.defaultParallelism)
print("Spark UI           :", sc.uiWebUrl)

### 1.3 Prueba mínima de RDD y DataFrame

In [ ]:
rdd_prueba = sc.parallelize(range(1, 11), 2)
print("RDD -> suma:", rdd_prueba.sum(), "| particiones:", rdd_prueba.getNumPartitions())

df_prueba = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "letra"])
df_prueba.show()
df_prueba.printSchema()

## 2. Lectura e inspección de los archivos de deportistas

Los datos de deportistas vienen repartidos en dos archivos CSV con la misma estructura de siete columnas:

| Columna | Contenido | Tipo |
|---|---|---|
| `deportista_id` | identificador del deportista (clave usada en `resultados.csv`) | entero |
| `nombre` | nombre del deportista | texto |
| `genero` | 1 = hombre, 2 = mujer | entero |
| `edad` | edad en años | entero |
| `altura` | altura en centímetros | decimal |
| `peso` | peso en kilogramos | decimal |
| `equipo_id` | identificador del equipo (clave de `equipo.csv`) | entero |

Antes de crear los RDDs se revisa el contenido crudo de cada archivo: separador, presencia de encabezado, uso de comillas y valores ausentes.

### 2.1 Ubicación de los archivos

In [ ]:
from pathlib import Path

DIR_DATOS = Path("data")
RUTA_DEPORTISTA = DIR_DATOS / "deportista.csv"
RUTA_DEPORTISTA2 = DIR_DATOS / "deportista2.csv"

for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    estado = f"{ruta.stat().st_size:,} bytes" if ruta.exists() else "NO ENCONTRADO"
    print(f"{str(ruta):22s} {estado}")

faltantes = [str(r) for r in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2) if not r.exists()]
if faltantes:
    raise FileNotFoundError(f"Faltan archivos de entrada en la carpeta data/: {faltantes}")

### 2.2 Contenido crudo

Se muestran las primeras líneas tal como las lee Spark con `textFile`. Se usa `repr` para dejar visibles las comillas y los caracteres de fin de línea.

In [ ]:
for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    lineas = sc.textFile(str(ruta))
    print(f"--- {ruta.name}: {lineas.count():,} líneas")
    for linea in lineas.take(5):
        print("   ", repr(linea))
    print()

### 2.3 Reglas de lectura

- **Separación de campos:** cada línea se divide con el lector CSV de Python, que respeta los nombres entre comillas aunque contengan comas.
- **Encabezado:** solo la primera línea de un archivo puede ser encabezado; se reconoce porque su primer campo no es numérico. Se retira y se informa.
- **Valores ausentes:** los campos vacíos o con `NA` se interpretan como desconocidos (`None`).
- **Campos vacíos sobrantes:** si una línea trae más de siete campos y los sobrantes están vacíos (coma final de más), se conservan los siete primeros. Estos casos se cuentan y se muestran.
- **Registros inválidos:** una línea con un número de campos distinto de siete o con valores numéricos imposibles de convertir no se descarta en silencio: se separa en un RDD de inválidos y se informa con su motivo.
- **Ceros:** se conservan tal como vienen en los RDDs para mantener la trazabilidad con la fuente; su tratamiento se define al construir el DataFrame.

In [ ]:
import csv

COLUMNAS = ["deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"]
VALORES_AUSENTES = {"", "NA", "NaN", "null", "None"}


def separar_campos(linea):
    """Divide una línea CSV respetando comillas (nombres que contienen comas)."""
    return next(csv.reader([linea.rstrip("\r")]))


def sobran_campos_vacios(campos):
    """True si la línea trae más de siete campos y los sobrantes están vacíos (coma final extra)."""
    return len(campos) > len(COLUMNAS) and all(c.strip() == "" for c in campos[len(COLUMNAS):])


def normalizar_campos(campos):
    """Elimina los campos vacíos sobrantes al final de la línea; el resto queda igual."""
    return campos[:len(COLUMNAS)] if sobran_campos_vacios(campos) else campos


def es_encabezado(campos):
    return len(campos) > 0 and not campos[0].strip().isdigit()


def a_entero(valor):
    valor = valor.strip()
    if valor in VALORES_AUSENTES:
        return None
    numero = float(valor)
    if not numero.is_integer():
        raise ValueError(f"'{valor}' no es entero")
    return int(numero)


def a_decimal(valor):
    valor = valor.strip()
    return None if valor in VALORES_AUSENTES else float(valor)


def convertir_registro(campos):
    """Devuelve (registro_tipado, None) si la línea es válida, o (campos, motivo) si no lo es."""
    if len(campos) != len(COLUMNAS):
        return campos, f"tiene {len(campos)} campos"
    try:
        d_id, nombre, genero, edad, altura, peso, equipo_id = campos
        registro = (a_entero(d_id), nombre.strip(), a_entero(genero), a_entero(edad),
                    a_decimal(altura), a_decimal(peso), a_entero(equipo_id))
    except ValueError as error:
        return campos, f"valor no numérico ({error})"
    if registro[0] is None:
        return campos, "deportista_id vacío"
    return registro, None


def leer_deportistas(lineas):
    """A partir de un RDD de líneas devuelve (validos, invalidos, encabezado)."""
    indexadas = lineas.zipWithIndex()
    encabezado = (indexadas
                  .filter(lambda par: par[1] == 0 and es_encabezado(separar_campos(par[0])))
                  .map(lambda par: par[0]))
    convertidos = (indexadas
                   .filter(lambda par: not (par[1] == 0 and es_encabezado(separar_campos(par[0]))))
                   .map(lambda par: convertir_registro(normalizar_campos(separar_campos(par[0])))))
    validos = convertidos.filter(lambda r: r[1] is None).map(lambda r: r[0])
    invalidos = convertidos.filter(lambda r: r[1] is not None)
    return validos, invalidos, encabezado

### 2.4 Diagnóstico de calidad de cada archivo

Para cada archivo se cuenta el encabezado, los registros válidos e inválidos, los valores ausentes y los ceros de las columnas numéricas, y la codificación de `genero`.

In [ ]:
def diagnosticar(ruta):
    lineas = sc.textFile(str(ruta))
    validos, invalidos, encabezado = leer_deportistas(lineas)
    validos.cache()

    total_lineas = lineas.count()
    n_encabezado = encabezado.count()
    n_validos = validos.count()
    n_invalidos = invalidos.count()

    print(f"=== {ruta.name}")
    print(f"Líneas leídas       : {total_lineas:,}")
    print(f"Encabezado          : {encabezado.first() if n_encabezado else 'no tiene'}")
    reparadas = lineas.filter(lambda l: sobran_campos_vacios(separar_campos(l)))
    print(f"Registros válidos   : {n_validos:,}")
    print(f"  con campos vacíos sobrantes reparados: {reparadas.count():,}")
    for linea in reparadas.take(5):
        print("    ", repr(linea))
    print(f"Registros inválidos : {n_invalidos:,}")
    for campos, motivo in invalidos.take(5):
        print(f"    {motivo}: {campos}")
    print(f"Conciliación        : {n_encabezado} + {n_validos:,} + {n_invalidos:,} = "
          f"{n_encabezado + n_validos + n_invalidos:,} ({'OK' if n_encabezado + n_validos + n_invalidos == total_lineas else 'NO CUADRA'})")

    ids = validos.map(lambda r: r[0])
    print(f"Rango deportista_id : {ids.min():,} a {ids.max():,} | ids distintos: {ids.distinct().count():,}")

    print(f"{'columna':10s} {'ausentes':>9s} {'ceros':>9s} {'mínimo':>8s} {'máximo':>8s}")
    for i, columna in enumerate(COLUMNAS):
        if columna in ("deportista_id", "nombre"):
            continue
        valores = validos.map(lambda r, i=i: r[i])
        presentes = valores.filter(lambda v: v is not None)
        print(f"{columna:10s} {valores.filter(lambda v: v is None).count():9,} "
              f"{presentes.filter(lambda v: v == 0).count():9,} {presentes.min():8} {presentes.max():8}")

    print("Codificación genero :", dict(sorted(validos.map(lambda r: r[2]).countByValue().items(),
                                              key=lambda kv: (kv[0] is None, kv[0]))))
    con_comillas = lineas.filter(lambda l: '"' in l).count()
    print(f"Líneas con comillas : {con_comillas:,}")
    validos.unpersist()
    print()


for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    diagnosticar(ruta)

**Hallazgos de la inspección**

| | `deportista.csv` | `deportista2.csv` |
|---|---|---|
| Líneas | 67.786 | 67.785 |
| Encabezado | no tiene | no tiene |
| Fin de línea | CRLF (Windows) | LF |
| Rango de `deportista_id` | 1 a 67.786 | 67.787 a 135.571 |
| Registros inválidos | 0 | 0 |
| Campos vacíos sobrantes reparados | 1 | 0 |
| Edad = 0 | 3.235 | 3.133 |
| Altura = 0 | 17.435 | 16.481 |
| Peso = 0 | 17.949 | 16.936 |
| `equipo_id` = 0 | 64 | 79 |
| Género 1 / 2 | 51.380 / 16.406 | 50.210 / 17.575 |

- Ninguno de los dos archivos trae encabezado y ninguno usa comillas; aun así, el lector respeta comillas por si un nombre contuviera comas. El fin de línea CRLF de `deportista.csv` se elimina al leer.
- Los dos archivos son complementarios: los identificadores son continuos del 1 al 135.571, sin repetidos ni vacíos entre ellos.
- La línea del deportista **16587** (`16587,Vladimr Bil,1,24,191,92,67,`) termina con una coma de más, que genera un octavo campo vacío. El registro está completo, por lo que se conservan sus siete campos en lugar de descartarlo.
- `genero` usa exclusivamente los códigos 1 (hombre) y 2 (mujer).
- Hay valores **0** en `edad`, `altura` y `peso`. Una edad, altura o peso igual a cero no es posible en un deportista: se trata de datos no registrados en la fuente. En los RDDs se conservan sin cambios y en el DataFrame se convierten a nulo.
- 143 deportistas tienen `equipo_id = 0`, código que no existe en `equipo.csv`; se mantienen y se informan en la integración con las demás tablas.
- Algunos nombres muestran caracteres perdidos en la fuente (por ejemplo, `Vladimr Bil` o `Piotr ya`, donde faltan letras con diacríticos); se conservan tal como vienen.

## 3. Creación de RDDs y unión

Se crea el RDD `deportista` a partir de `deportista.csv` con **seis particiones** y el RDD `deportista2` a partir de `deportista2.csv`. Luego se unen en `deportistaTotal`.

`textFile` recibe un número **mínimo** de particiones; si Spark genera más de seis, se reducen con `coalesce(6)`, que no redistribuye los datos entre nodos.

In [ ]:
lineas_deportista = sc.textFile(str(RUTA_DEPORTISTA), minPartitions=6)
deportista, invalidos_deportista, encabezado_deportista = leer_deportistas(lineas_deportista)
if deportista.getNumPartitions() > 6:
    deportista = deportista.coalesce(6)
deportista = deportista.cache()

print("RDD deportista")
print("  Particiones          :", deportista.getNumPartitions())
print("  Registros por partición:", deportista.glom().map(len).collect())
print(f"  Registros válidos    : {deportista.count():,}")
print(f"  Registros inválidos  : {invalidos_deportista.count():,}")
print("  Muestra:")
for registro in deportista.take(5):
    print("   ", registro)

In [ ]:
lineas_deportista2 = sc.textFile(str(RUTA_DEPORTISTA2))
deportista2, invalidos_deportista2, encabezado_deportista2 = leer_deportistas(lineas_deportista2)
deportista2 = deportista2.cache()

print("RDD deportista2")
print("  Particiones          :", deportista2.getNumPartitions())
print(f"  Registros válidos    : {deportista2.count():,}")
print(f"  Registros inválidos  : {invalidos_deportista2.count():,}")
print("  Muestra:")
for registro in deportista2.take(5):
    print("   ", registro)

### 3.1 Unión en `deportistaTotal`

`union` concatena ambos RDDs sin eliminar duplicados, por lo que el número de particiones del resultado es la suma de las particiones de ambos. Se verifica que el total cuadre con la suma de los registros válidos de cada archivo y que ningún `deportista_id` se repita entre los dos archivos.

In [ ]:
deportistaTotal = deportista.union(deportista2).cache()

n_deportista = deportista.count()
n_deportista2 = deportista2.count()
n_total = deportistaTotal.count()
n_ids_distintos = deportistaTotal.map(lambda r: r[0]).distinct().count()

print("RDD deportistaTotal")
print("  Particiones              :", deportistaTotal.getNumPartitions(),
      f"({deportista.getNumPartitions()} + {deportista2.getNumPartitions()})")
print(f"  deportista + deportista2 : {n_deportista:,} + {n_deportista2:,} = {n_deportista + n_deportista2:,}")
print(f"  deportistaTotal          : {n_total:,}")
print(f"  deportista_id distintos  : {n_ids_distintos:,}")

assert n_total == n_deportista + n_deportista2, "La unión no conserva todos los registros"
print("  Conciliación             : OK" if n_ids_distintos == n_total
      else f"  Atención: {n_total - n_ids_distintos:,} identificadores repetidos entre archivos")

## 4. Transformaciones sobre RDDs

Las transformaciones se aplican sobre `deportistaTotal` con operaciones propias de RDD (`filter`, `map`), sin pasar por DataFrames. Cada registro es una tupla con el orden de `COLUMNAS`: la edad está en la posición 3 y el género en la posición 2.

### 4.1 `MayorEdad`: deportistas de 18 años o más

Una edad igual a 0 o ausente corresponde a un dato no registrado; esos deportistas no se cuentan como mayores ni como menores de edad, y se informan por separado.

In [ ]:
MayorEdad = deportistaTotal.filter(lambda r: r[3] is not None and r[3] >= 18)

edad_desconocida = deportistaTotal.filter(lambda r: r[3] is None or r[3] == 0)
menores = deportistaTotal.filter(lambda r: r[3] is not None and 0 < r[3] < 18)

n_mayores, n_menores, n_desconocida = MayorEdad.count(), menores.count(), edad_desconocida.count()

print(f"Mayores de edad (>= 18) : {n_mayores:,}")
print(f"Menores de edad (1-17)  : {n_menores:,}")
print(f"Edad no registrada      : {n_desconocida:,}")
print(f"Suma                    : {n_mayores + n_menores + n_desconocida:,} de {n_total:,}")
print("Edad mínima en MayorEdad:", MayorEdad.map(lambda r: r[3]).min())
print("Muestra:")
for registro in MayorEdad.take(5):
    print("   ", registro)

### 4.2 `Deportistas_mujer`: deportistas con género 2

Según la codificación de la fuente, el valor 2 de `genero` corresponde a mujer.

In [ ]:
Deportistas_mujer = deportistaTotal.filter(lambda r: r[2] == 2)

n_mujeres = Deportistas_mujer.count()
n_hombres = deportistaTotal.filter(lambda r: r[2] == 1).count()

print(f"Mujeres (genero = 2)        : {n_mujeres:,}")
print(f"Hombres (genero = 1)        : {n_hombres:,}")
print(f"Otro código o ausente       : {n_total - n_mujeres - n_hombres:,}")
print("Valores de genero en el RDD :", sorted(Deportistas_mujer.map(lambda r: r[2]).distinct().collect()))
print("Muestra:")
for registro in Deportistas_mujer.take(5):
    print("   ", registro)

### 4.3 Conversión a mayúsculas

Se convierten a mayúsculas los campos de texto de `deportistaTotal`; en este conjunto el único campo textual es `nombre`. Los valores numéricos no se modifican.

Antes de reasignar el nombre se guarda una referencia al RDD original (`deportistaTotal_original`) para comparar ambos y mantener la trazabilidad. Como `map` conserva el orden y las particiones, los registros de ambos RDDs pueden emparejarse con `zip`.

In [ ]:
deportistaTotal_original = deportistaTotal

deportistaTotal = deportistaTotal_original.map(
    lambda r: tuple(v.upper() if isinstance(v, str) else v for v in r)
).cache()

print(f"{'deportista_id':>13s}  {'nombre original':35s} nombre en mayúsculas")
for original, mayus in deportistaTotal_original.zip(deportistaTotal).take(10):
    print(f"{original[0]:>13}  {original[1]:35s} {mayus[1]}")

pares = deportistaTotal_original.zip(deportistaTotal)
numericos_alterados = pares.filter(lambda p: p[0][0] != p[1][0] or p[0][2:] != p[1][2:]).count()
sin_minusculas = deportistaTotal.filter(lambda r: r[1] != r[1].upper()).count()

print()
print(f"Registros en deportistaTotal       : {deportistaTotal.count():,}")
print(f"Nombres que aún tienen minúsculas  : {sin_minusculas:,}")
print(f"Registros con valores numéricos alterados: {numericos_alterados:,}")

## 5. Conversión a DataFrame `deportista`

`deportistaTotal` (ya en mayúsculas) se convierte en el DataFrame `deportista` con las siete columnas y un esquema explícito, de modo que cada columna tenga un tipo numérico o de texto utilizable en las etapas siguientes.

- `edad`, `altura` y `peso` iguales a 0 se convierten a **nulo**, porque representan datos no registrados. Así no distorsionan promedios, mínimos ni el cálculo de IMC.
- `genero` se mantiene con su codificación original (1 = hombre, 2 = mujer).
- `equipo_id` se mantiene sin cambios, incluido el valor 0, para revisarlo en la integración con `equipo.csv`.

El nombre `deportista` pasa a designar el DataFrame; el RDD original se conserva en `rdd_deportista`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

rdd_deportista = deportista   # referencia al RDD original antes de reutilizar el nombre

esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("genero", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("altura", DoubleType(), True),
    StructField("peso", DoubleType(), True),
    StructField("equipo_id", IntegerType(), True),
])


def cero_a_nulo(valor):
    return None if valor == 0 else valor


filas = deportistaTotal.map(
    lambda r: (r[0], r[1], r[2], cero_a_nulo(r[3]), cero_a_nulo(r[4]), cero_a_nulo(r[5]), r[6])
)
deportista = spark.createDataFrame(filas, schema=esquema_deportista)

print(f"Filas del DataFrame deportista: {deportista.count():,} (deportistaTotal: {deportistaTotal.count():,})")
print("Columnas:", deportista.columns)
deportista.printSchema()
deportista.show(10, truncate=False)

### 5.1 Valores nulos por columna

In [ ]:
deportista.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in deportista.columns]
).show()

deportista.describe("edad", "altura", "peso").show()

### 5.2 Resumen

| Elemento | Resultado |
|---|---|
| RDD `deportista` | 67.786 registros de `deportista.csv` en seis particiones de entre 11.095 y 11.424 registros |
| RDD `deportista2` | 67.785 registros de `deportista2.csv` |
| RDD `deportistaTotal` | 135.571 registros (67.786 + 67.785), sin identificadores repetidos; nombres en mayúsculas |
| RDD `MayorEdad` | 123.244 deportistas de 18 años o más |
| RDD `Deportistas_mujer` | 33.981 deportistas con `genero = 2` |
| DataFrame `deportista` | 135.571 filas y siete columnas tipadas |

**Conclusiones**

- La unión conserva todos los registros de ambos archivos: 135.571 deportistas con identificadores únicos y continuos, que corresponden a los `deportista_id` usados en `resultados.csv`.
- Del total, 123.244 deportistas son mayores de edad, 5.959 tienen entre 10 y 17 años y 6.368 no tienen edad registrada. Estos últimos no se clasifican como menores.
- Las mujeres representan 33.981 registros (25,1 %) y los hombres 101.590 (74,9 %).
- La conversión a mayúsculas afectó solo a los nombres: ningún valor numérico cambió.
- En el DataFrame quedan sin dato 6.368 edades (4,7 %), 33.916 alturas (25,0 %) y 34.885 pesos (25,7 %). Entre los valores registrados, la edad promedio es de 24,4 años (rango de 10 a 97), la altura promedio de 176,3 cm y el peso promedio de 72,0 kg.
- La falta de altura o peso en cerca de un cuarto de los deportistas limitará el cálculo del IMC a los registros que tengan ambos valores.